# Lab 5.3 &mdash; The Agent Loop: Reason, Act, Observe

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Run the reason -> act -> observe cycle, driven by a REAL model
- Stop when the model returns a Final Answer
- Cap the loop with max_steps so a confused model can never run forever

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. In Module 5 you build the agent **from scratch** &mdash; the loop, the ReAct parser, the tool router, the memory, the guardrails &mdash; as **real Python**. What's new vs the old version: a **real local model** now does the *reasoning* (it emits the ReAct steps / picks the actions) while **your** code parses, routes and loops it, and **real tools** run. Read the **Concept**, fill the real `___` blanks in **Build it**, then **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**.

> **Framework note:** these labs use a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`) via `langchain-ollama`. Unlike Module 6, you do **not** hand the loop to a framework &mdash; you build it yourself and the model drives it. If Ollama is down, the run cells print how to start it instead of crashing. A tool must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole loop (you'll see exactly this in Labs 5 and 8). A small model can pick a wrong tool or fumble a step now and then &mdash; that's real agent behaviour, and it's why you read the trace.

**Reference:** [Module 5 slides &mdash; The agent loop](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"))   # GROQ/OPENAI keys, if you ever want a hosted alternative

WORK = "/tmp/biaa-lab-05-03"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model DRIVES the agent YOU build from scratch.
# Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def llm_text(prompt):
    """Call the REAL model and return its text (the .content of the reply)."""
    return llm.invoke(prompt).content

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Every agent runs the same cycle: **Reason** (decide the next step), **Act** (call a tool), **Observe**
(read the result), then **repeat** &mdash; until it can **stop** and answer. That is **ReAct** (Reason +
Act). Here the **Reason** step is a **real `llama3.1:8b`** emitting ReAct text; **your** loop parses it,
runs the tool, feeds the observation back, and stops on a Final Answer or a **max_steps** cap. The
`react_prompt`, `parse_step` and `run_tool` plumbing is provided &mdash; you wire the **loop**.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

KNOWLEDGE = {"population of metropolis": "120000", "capital of france": "Paris"}
def _lookup(key):
    return KNOWLEDGE.get(str(key).lower().strip(), "unknown")
def _calc(expr):
    try:
        return str(safe_calc(expr))
    except Exception:
        return "error: not a numeric expression"   # a tool must NEVER raise -- return a string
TOOLS = {"lookup": _lookup, "calculator": _calc}    # real from-scratch tools: name -> function

def run_tool(name, inp):
    """Route an action to a tool and return an OBSERVATION string -- never raises."""
    fn = TOOLS.get(str(name).strip().lower())
    if fn is None:
        return "unknown tool: " + str(name)
    arg = str(inp).strip().strip("'\"").strip()   # tolerate a model that quotes its argument
    return fn(arg)

REACT_SYS = (
    "You are a ReAct agent. Solve the task step by step using tools.\n"
    "Tools:\n"
    "- lookup: look up a known fact by key. Known keys: 'population of metropolis', 'capital of france'.\n"
    "- calculator: do exact arithmetic, e.g. 120000*2.\n"
    "Each step you output is EXACTLY:\n"
    "Thought: <reasoning>\n"
    "Action: lookup OR calculator\n"
    "Action Input: <input to that tool>\n"
    "You will then be shown an 'Observation:'. Use it to decide the next step.\n"
    "When the observations are enough to answer, output instead:\n"
    "Thought: <reasoning>\n"
    "Final Answer: <the answer>\n"
    "Give ONLY the next single step. Do NOT write 'Observation:' yourself."
)

def react_prompt(task, transcript):
    """The prompt sent to the REAL model each turn: the format spec + task + the transcript so far.
    The transcript keeps the model's OWN Thoughts/Actions + the tool Observations, so it can continue
    its reasoning instead of restarting."""
    return REACT_SYS + "\n\nTask: " + task + "\n" + transcript

def parse_step(text):
    """Parse the model's REAL text into ('final', answer) | ('action', name, inp) | ('none', text)."""
    def field(label):
        for line in str(text).splitlines():
            if line.strip().lower().startswith(label.lower() + ":"):
                return line.split(":", 1)[1].strip()
        return None
    fin = field("Final Answer")
    if fin is not None:
        return ("final", fin)
    act = field("Action")
    if act:
        return ("action", act.strip().lower(), field("Action Input") or "")
    return ("none", str(text)[:80])

print("tools:", list(TOOLS))
print("run_tool('lookup', 'population of metropolis') ->", run_tool("lookup", "population of metropolis"))
print("parse_step demo ->", parse_step("Thought: t\nAction: calculator\nAction Input: 2*3"))

## Build it
Complete `run_loop`: each turn, ask the **real** model for a step, **stop** on a Final Answer, otherwise
**run** the tool and append the observation. (Labs 4 and 5 dig into the parser and the router you're using
here.)

In [ ]:
def run_loop(task, max_steps=5):
    transcript, trace = "", []
    for step in range(max_steps):
        text = ___                       # TODO: ask the REAL model for the next step: react_prompt(task, transcript)
        kind = parse_step(text)
        if ___:                          # TODO: stop when the model gave a Final Answer (kind[0] == "final")
            return {"answer": kind[1], "trace": trace, "steps": step + 1}
        if kind[0] == "action":
            name, inp = kind[1], kind[2]
            obs = ___                    # TODO: run the chosen tool SAFELY: run_tool(name, inp)
            trace.append((name, inp, obs))
            transcript += text.strip() + "\nObservation: " + obs + "\n"
        else:
            transcript += text.strip() + "\nObservation: (unparseable step)\n"
    return {"answer": None, "trace": trace, "steps": max_steps}

try:
    print("prompt preview:\n", react_prompt("twice the population of metropolis", "")[:220], "...")
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Run the loop on a two-step task. The real model reasons; your loop calls `lookup` then `calculator`. Read the trace.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        out = run_loop("What is twice the population of metropolis?")
        print("TRACE (a REAL model + YOUR loop):")
        for name, inp, obs in out["trace"]:
            print("  ACTION:", name, "| INPUT:", inp, "-> OBS:", obs)
        print("steps:", out["steps"], "| ANSWER:", out["answer"])
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Each `ACTION` line was **chosen by the real model** in ReAct text; your loop parsed it and ran your Python tool.
- **`max_steps`** is your one-line guardrail &mdash; even a model that never says "Final Answer" can't loop forever.
- If the model computed without looking up, or chose a different order, that's **real 8b behaviour** &mdash; the trace tells you the truth.

## Your turn (open task &mdash; no grader)
Change the task (e.g. *"what is the capital of france?"*, which needs only `lookup`) or add a tool to `TOOLS`
and mention it in `REACT_SYS`, then re-run. **What good looks like:** the trace shows the model calling the
right tool(s) and a sensible Final Answer; a task it can't do stops cleanly at the cap.

---
### Key takeaway
Reason -> act -> observe, repeat, with a stop condition and a step cap. That loop IS the agent -- and a REAL model is now the reasoner driving your tools.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>